# Final project group 17

For the final project we are required make a program to create a map 
to calibrate the cameras with images 0_calib

## Project goals

1. Calibrate the stereo camera system and rectify the input images.
2. Then, for each sequence, do the following:
i. Compute disparity maps and reconstruct the 3D structure of the scene from the rectified stereo
image pairs.
ii. Detect the Husky A200 robot in the scene by processing the input images.
iii. Track the Husky A200 robot over time in 2D and 3D, including estimating its position during
complete occlusions

### Story
At ASTA (Autonomous Systems Testing Arena), our robotics testing facility at DTU, our mobile robot,
Husky, has been assigned to transport tools and equipment between testing stations. However,
we suspect that Husky has been taking some rather unusual routes instead of going directly to its
destinations. Luckily, two cameras have been recording its movements around ASTA, and we have
obtained footage of four of its recent deliveries. Your task is to calibrate the cameras and develop a
perception system that tracks Husky’s movements and reconstructs its trajectory in 2D and 3D, even
when it temporarily disappears from view

### Initialization

In [2]:
# Initialization

# calibration board ~ 80mm x 80mm

import numpy as np
import cv2 as cv2
import glob
import matplotlib.pyplot as plt

### Load image paths
# calibration
left_img_calib = sorted(glob.glob('0_calib/left/*.png'))
right_img_calib = sorted(glob.glob('0_calib/right/*.png'))

# no obstacle
left_img_no_obs = sorted(glob.glob('1_no_obstacle/left/*.png'))
right_img_no_obs = sorted(glob.glob('1_no_obstacle/right/*.png'))

# obstacle
left_img_obs = sorted(glob.glob('2_obstacle/left/*.png'))
right_img_obs = sorted(glob.glob('2_obstacle/right/*.png'))

# curved bend
left_img_curve = sorted(glob.glob('3_curved_bend/left/*.png'))
right_img_curve = sorted(glob.glob('3_curved_bend/right/*.png'))

# circle
left_img_circle = sorted(glob.glob('4_circle/left/*.png'))
right_img_circle = sorted(glob.glob('4_circle/right/*.png'))

### Camera calibration

In [10]:
print(objp_left*80)

[[  0.   0.   0.]
 [ 80.   0.   0.]
 [160.   0.   0.]
 [240.   0.   0.]
 [320.   0.   0.]
 [400.   0.   0.]
 [480.   0.   0.]
 [  0.  80.   0.]
 [ 80.  80.   0.]
 [160.  80.   0.]
 [240.  80.   0.]
 [320.  80.   0.]
 [400.  80.   0.]
 [480.  80.   0.]
 [  0. 160.   0.]
 [ 80. 160.   0.]
 [160. 160.   0.]
 [240. 160.   0.]
 [320. 160.   0.]
 [400. 160.   0.]
 [480. 160.   0.]
 [  0. 240.   0.]
 [ 80. 240.   0.]
 [160. 240.   0.]
 [240. 240.   0.]
 [320. 240.   0.]
 [400. 240.   0.]
 [480. 240.   0.]
 [  0. 320.   0.]
 [ 80. 320.   0.]
 [160. 320.   0.]
 [240. 320.   0.]
 [320. 320.   0.]
 [400. 320.   0.]
 [480. 320.   0.]]


In [12]:
nb_vertical = 7
nb_horizontal = 5

#cv2.CALIB_CB_ADAPTIVE_THRESH 
#                    + cv2.CALIB_CB_NORMALIZE_IMAGE

board_size = (80, 80) # mm
# Prepare object points, like (0, 0, 0), (1, 0, 0), (2, 0, 0) ....,(6, 5, 0).
objp_left = np.zeros((nb_horizontal * nb_vertical, 3), np.float32)*80
objp_left[:, :2] = np.mgrid[0:nb_vertical, 0:nb_horizontal].T.reshape(-1, 2)

objp_right = np.zeros((nb_horizontal * nb_vertical, 3), np.float32)*80
objp_right[:, :2] = np.mgrid[0:nb_vertical, 0:nb_horizontal].T.reshape(-1, 2)

# Arrays to store object points and image points from all the images.
objpoints_left = []  # 3D point in real world space.
imgpoints_left = []  # 2D points in image plane.

objpoints_right = []  # 3D point in real world space.
imgpoints_right = []  # 2D points in image plane.

criteria = (cv2.TERM_CRITERIA_EPS + 
            cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

assert left_img_calib

for i, fname in enumerate(left_img_calib):
    img = cv2.imread(fname, 0)
    h, w = img.shape[:2]
    # flag cv2.CALIB_CB_ADAPTIVE_THRESH 
    ret, corners = cv2.findChessboardCorners(
        img, (nb_vertical, nb_horizontal), None
    )

    # If found, add object points, image points (after refining them).
    if ret == True:
        objpoints_left.append(objp_left)

        # first (15,15)
        corners2 = cv2.cornerSubPix(
            img, corners, (9, 9), (-1, -1), criteria)
        
        imgpoints_left.append(corners2)

        # Draw and display the corners.
        img = cv2.drawChessboardCorners(
            img, (nb_vertical, nb_horizontal), corners2, ret
        )
        cv2.imshow("img", img)
        cv2.imwrite('chessboard/flag/left/img{}.jpg'.format(i), img)
        cv2.waitKey(50)

assert right_img_calib

for i, fname in enumerate(right_img_calib):
    img = cv2.imread(fname, 0)
    h, w = img.shape[:2]

    ret, corners = cv2.findChessboardCorners(
        img, (nb_vertical, nb_horizontal), None
    )

    # If found, add object points, image points (after refining them).
    if ret == True:
        objpoints_right.append(objp_right)

        corners2 = cv2.cornerSubPix(
            img, corners, (9, 9), (-1, -1), criteria)
        
        imgpoints_right.append(corners2)

        # Draw and display the corners.
        img = cv2.drawChessboardCorners(
            img, (nb_vertical, nb_horizontal), corners2, ret
        )
        cv2.imshow("img", img)
        cv2.imwrite('chessboard/flag/right/img{}.jpg'.format(i), img)
        cv2.waitKey(50)

cv2.destroyAllWindows()

In [110]:
img = cv2.imread(left_img_calib[0], 0)
h, w = img.shape[:2]

ret_left, mtx_left, dist_left, rvecs, tvecs = cv2.calibrateCamera(
    objpoints_left, imgpoints_left, img.shape[::-1], None, None)

newcameramtx_left, roi_left = cv2.getOptimalNewCameraMatrix(mtx_left, dist_left, (w, h), 1, (w, h))

ret_right, mtx_right, dist_right, rvecs, tvecs = cv2.calibrateCamera(
    objpoints_right, imgpoints_right, img.shape[::-1], None, None
)
newcameramtx_right, roi_right = cv2.getOptimalNewCameraMatrix(mtx_right, dist_right, (w, h), 1, (w, h))

New camera matrix left:
[[693.07345827   0.         561.88055535]
 [  0.         675.45573083 326.08398008]
 [  0.           0.           1.        ]]
New camera matrix right
[[515.14418378   0.         698.9154794 ]
 [  0.         516.46543522 434.63381535]
 [  0.           0.           1.        ]]
Camera matrix left:
[[683.08651006   0.         596.88718862]
 [  0.         683.62460949 334.49430884]
 [  0.           0.           1.        ]]
Camera matrix right:
[[604.82866338   0.         654.01117341]
 [  0.         607.97415499 401.8124019 ]
 [  0.           0.           1.        ]]
Distortion coefficients left:
[[ 0.05970896 -0.07602472 -0.01044058 -0.02398194  0.05266709]]
Distortion coefficients right:
[[-0.07206082  0.06665457  0.00596237  0.00288189 -0.02923708]]


In [ ]:
print("New camera matrix left:")
print(newcameramtx_left)
print("New camera matrix right")
print(newcameramtx_right)

print("Camera matrix left:")
print(mtx_left)
print("Camera matrix right:")
print(mtx_right)

print("Distortion coefficients left:")
print(dist_left)
print("Distortion coefficients right:")
print(dist_right)

### Rectify and save images

In [119]:
# sanity check

seq = 20
img_l = cv2.imread(left_img_calib[seq], 0)
img_r = cv2.imread(right_img_calib[seq], 0)
image_size = img.shape[::-1]  # (w, h)

# undistort
dst_left = cv2.undistort(img_l, mtx_left, dist_left, None, newcameramtx_left)
 
# crop the image
x, y, w, h = roi_left
dst_left = dst_left[y:y+h, x:x+w]
cv2.imwrite('calibresult_l.png', dst_left)

# undistort
dst_right = cv2.undistort(img_r, mtx_right, dist_right, None, newcameramtx_right)
 
# crop the image
x, y, w, h = roi_right
dst_right = dst_right[y:y+h, x:x+w]
cv2.imwrite('calibresult_r.png', dst_right)

True

In [33]:
k = 0
for i, fname in enumerate(left_img_calib):
    test = cv2.imread(fname)
    test = cv2.cvtColor(test, cv2.COLOR_BGR2GRAY)
    #cv2.imshow('test', test)
    ret, corners = cv2.findChessboardCorners(
            test, (nb_vertical, nb_horizontal), None
    )
    print('img {} is {}'.format(i, ret))
    if ret == True:
        k += 1
print('successful: {}'.format(k))
#cv2.waitKey(10000)
cv2.destroyAllWindows()

img 0 is True
img 1 is True
img 2 is True
img 3 is True
img 4 is True
img 5 is True
img 6 is True
img 7 is True
img 8 is True
img 9 is True
img 10 is True
img 11 is True
img 12 is True
img 13 is False
img 14 is True
img 15 is True
img 16 is True
img 17 is True
img 18 is True
img 19 is False
img 20 is False
img 21 is False
img 22 is True
img 23 is True
img 24 is False
img 25 is True
img 26 is False
img 27 is False
img 28 is False
img 29 is False
img 30 is True
img 31 is False
img 32 is False
img 33 is True
img 34 is True
img 35 is True
img 36 is False
successful: 25


In [22]:
help(cv2.cvtColor)

Help on built-in function cvtColor:

cvtColor(...)
    cvtColor(src, code[, dst[, dstCn[, hint]]]) -> dst
    .   @brief Converts an image from one color space to another.
    .   
    .   The function converts an input image from one color space to another. In case of a transformation
    .   to-from RGB color space, the order of the channels should be specified explicitly (RGB or BGR). Note
    .   that the default color format in OpenCV is often referred to as RGB but it is actually BGR (the
    .   bytes are reversed). So the first byte in a standard (24-bit) color image will be an 8-bit Blue
    .   component, the second byte will be Green, and the third byte will be Red. The fourth, fifth, and
    .   sixth bytes would then be the second pixel (Blue, then Green, then Red), and so on.
    .   
    .   The conventional ranges for R, G, and B channel values are:
    .   -   0 to 255 for CV_8U images
    .   -   0 to 65535 for CV_16U images
    .   -   0 to 1 for CV_32F images
    . 

In [124]:
np.savetxt('roi_left', roi_left, delimiter = ', ')
np.savetxt('roi_right', roi_right, delimiter = ', ')
np.savetxt('mtx_left', mtx_left, delimiter = ', ')
np.savetxt('mtx_right', mtx_right, delimiter = ', ')
np.savetxt('dist_right', dist_right, delimiter = ', ')
np.savetxt('dist_left', dist_left, delimiter = ', ')